# Iterator Design Pattern

In [1]:
# from collections import defaultdict
from collections.abc import Iterable, Iterator, Sequence
from contextlib import suppress
from typing import Any, NoReturn, Self, TypeVar

## Intro

Iterator is a behavioural design pattern. It allows to traverse a collection without revealing the internal structure of the latter. Instead of of embedding possible ways to iterate over a collection, you may define iterators each responsible for a certain traversion logic.

In Python there are [iterable](https://docs.python.org/3/glossary.html#term-iterable) objects that are capable of returning its members one at a time.
Examples are all sequence types like [list](https://docs.python.org/3/library/stdtypes.html#list), [str](https://docs.python.org/3/library/stdtypes.html#str) and some non-sequence types like [dict](https://docs.python.org/3/library/stdtypes.html#dict), [file(-like) objects](https://docs.python.org/3/glossary.html#term-file-object) and so forth.

A class can be iterable if:

1. inherits from the [Iterable](https://docs.python.org/3/library/collections.abc.html#collections.abc.Iterable) collection
2. defines the [__iter__()](https://docs.python.org/3/library/stdtypes.html#container.__iter__) method which must return an [iterator](https://docs.python.org/3/glossary.html#term-iterator) object which, in its turn, is responsible for conveying the items from the given iterable object.

Let's just see it in use.

In [2]:
class SubIterable(Iterable):
    def __iter__(self) -> Self:
        return self

class SupportIter:
    def __iter__(self) -> Self:
        return self

sub = SubIterable()
sup = SupportIter()

# The objects are Iterable -> they do have `__iter__()` defined
assert isinstance(sub, Iterable)
assert isinstance(sup, Iterable)

# Because the defined `__iter__()` methods does not return iterators,
# calling `iter(obj)` failes.

with suppress(TypeError):
    iter(sub)  # like sub.__iter__()

with suppress(TypeError):
    iter(sup)

# Because `iter(obj)` does not return an iterator,
# such an object cannot be used in a `for` loop.

with suppress(TypeError):
    for _ in sub:
        break

with suppress(TypeError):
    for _ in sup:
        break

An interesting situation we have: objects are iterable, but they cannot be used properly in a [for](https://docs.python.org/3/reference/compound_stmts.html#for) loop. A key to understand the problem is to remember that a `for` loop is roughly equivalent to the following structure:

In [3]:
seq = (1, 2, 3, 4)  # a tuple is a Sequence

for i in seq:
    print(f"Item = {i}")
print()

# The above is something like

iterator = iter(seq)
print(
    f"Iterator = {iterator}:",
    "\n".join(
        [
            f"type={type(iterator)}",
            f"is_iterable={isinstance(iterator, Iterable)}",
            f"has __iter__? {hasattr(iterator, '__iter__')}",
            f"has __next__? {hasattr(iterator, '__next__')}",
        ]
    ),
    sep="\n",
    end="\n\n",
)

while True:
    try:
        item = next(iterator)  # like iterator.__next__()
    except StopIteration as e:
        print(f"Iteration is over: {e}")
        break

    print(f"Item = {i}")

Item = 1
Item = 2
Item = 3
Item = 4

Iterator = <tuple_iterator object at 0x7946e0144af0>:
type=<class 'tuple_iterator'>
is_iterable=True
has __iter__? True
has __next__? True

Item = 4
Item = 4
Item = 4
Item = 4
Iteration is over: 


So:

1. We need to get an iterator object from the associated iterable object. The returned iterator object is "Iterable" itself, i.e. it also has the **__iter__()** method defined.
2. A successive item is retrieved via the  **__next__()** method of the iterator object. The [__next__()](https://docs.python.org/3/library/stdtypes.html#iterator.__next__) is responsible for returning the next item from the iterator and when there are no further items, it raises the [StopIteration](https://docs.python.org/3/library/exceptions.html#StopIteration) exception which signals that the iteration is over.

Let's fix the previous custom iterable classes.

In [4]:
class MyCounter(Iterable):
    """This class is Iterable and Iterator per se."""

    def __init__(self, start: int = 5) -> None:
        self._countdown = start

    def __iter__(self) -> Self:
        # our iterator is reusable
        self._counter = self._countdown
        return self

    def __next__(self) -> int:
        self._counter -= 1
        if self._counter < 0:
            # the message is not necessary
            msg = f"EOC: {self._counter}"
            # raising just StopIteration is enough
            raise StopIteration(msg)
        return self._counter


counter = MyCounter(start=5)

assert isinstance(counter, Iterable)
assert isinstance(counter, Iterator)

iter(counter)  # OK

# does these `iter` and `next` calls for us
for c in counter:
    print(f"count = {c}")
print()

# # checking that the MyCounter is resuable 
# for c in counter:
#     print(f"count = {c}")
# print()

iterator = iter(counter)
while True:
    try:
        c = next(counter)
    except StopIteration as e:
        print(f"Stop: message={e}")
        break
    print(f"count = {c}")

count = 4
count = 3
count = 2
count = 1
count = 0

count = 4
count = 3
count = 2
count = 1
count = 0
Stop: message=EOC: -1


Or we can separate the concerns:

1. Define a class, so it be A, with **__iter__()** and this is our "Iterable" entity;
2. The **iter(A)** will return an instance of the class B which has only **__next__()**

Will it work?

In [5]:
class MyIterator:
    def __init__(self, seq: list) -> None:
        self._seq = seq
        self._indices = iter(range(0, len(seq), 2))

    def __next__(self):
        return self._seq[next(self._indices)]

class MyIterable:
    def __init__(self, it: Iterable | None = None) -> None:
        self._lst = list(it or [])

    def __iter__(self) -> MyIterator:
        return MyIterator(seq=self._lst)


it = MyIterable(range(10))
for item in it:
    print(f"item = {item}")


assert isinstance(it, Iterable)
# We have defined an iterator which IS NOT (!) Iterable
assert not isinstance(iter(it), Iterable)

# This assertion demonstrates that Iterator must be Iterable
assert not isinstance(iter(it), Iterator)

# MyIterator does not implement the `__iter__`,
# so it is impossible for it to be used in a for loop
with suppress(TypeError):
    for _ in MyIterator([]):
        break

item = 0
item = 2
item = 4
item = 6
item = 8


It works and the results are fascinating!

The "MyIterable" is suitable for a `for` loop -> no TypeError, no other complaints, iterations go fine. But we have to examine the assertions which tell us that:

1. "MyIterable" IS "Iterable" by implementing the **__iter__()** method.
2. "MyIterator" IS NOT "Iterable" because the former does not implement the **__iter__()** - it is fine, maybe.
3. "MyIterator" IS NOT "Iterator" and this is creepy, but understandable -> defining only the **__next__()** method is not enough to be "Iterator".

So the conclusion is that `Iterator` IS `Iterable`, therefore, an instance of "Iterator" class must implement both the **__iter__()** and **__next__()** methods to comply with this type hierarchy.

In [6]:
class DummyIterator:
    def __iter__(self) -> Self:
        return self

    def __next__(self) -> NoReturn:
        raise StopIteration

diter = DummyIterator()

assert isinstance(diter, Iterable)
assert isinstance(diter, Iterator)

assert issubclass(Iterator, Iterable)

Summary:

1. An object is iterable if it implements the **__iter__()** (and by this it is "Iterable") method that must return an iterator object that must define the **__next__()** method -> this will work, but if an iterator object does not specify its own **__iter__()** method, then such an instance is neither "Iterator" nor "Iterable".
2. If `iter(obj)` does not fail that its return value is an iterator object and, therefore, can be used in a `for` loop.
3. I think that one reason to force the **__iter__()** method for supporting `Iterator` protocol (by inheritance or by defining the corresponding magic methods) is to enable iterators to be used to a `for` loop per se because the latter calls (all under the hood) `iter` first and then `next` until `StopIteration`.

Iterable-Iterator relationship (Plant)UML diagramm:

![Iterable-Iterator](./Iterable-Iterator.png)

## Sequences

The objects you can iterate over may be defined not only through the **__iter__()/__next__()** pair. If a class defines the [__getitem__()](https://docs.python.org/3/reference/datamodel.html#object.__getitem__) and [__len__()](https://docs.python.org/3/reference/datamodel.html#object.__len__) dunder (double underscore) methods, then it is considered as a [sequence](https://docs.python.org/3/glossary.html#term-sequence) and thus becomes an iterable object.

In [7]:
class DummySequence:
    def __init__(self, seq: Iterable) -> None:
        self._seq = tuple(seq)

    def __getitem__(self, key: int | slice) -> Any:
        res = self._seq[key]
        if isinstance(key, int):
            return res
        return type(self)(res)

    def __len__(self) -> int:
        return len(self._seq)

    def __repr__(self) -> str:
        return f"{type(self).__name__}(seq={self._seq})"

seq = DummySequence([1, 4, 6, 9])

print(f"seq[1::2] = {seq[1::2]}")
print(f"len(seq) = {len(seq)}")

assert not isinstance(seq, Iterable)  # no __iter__

# https://docs.python.org/3/glossary.html#term-sequence
# https://stackoverflow.com/questions/64654517/why-does-isinstance-check-for-abc-sequence-return-false-for-custom-classes
assert not isinstance(seq, Sequence)

for elem in seq:
    print(f"Element = {elem}")

seq[1::2] = DummySequence(seq=(4, 9))
len(seq) = 4
Element = 1
Element = 4
Element = 6
Element = 9


How about `isinstance(obj, Sequence)` check? A way to discouver it by example is to define a custom class tht inherits from the `collections.abc.Sequence` class. The latter has abstract methods which must be overriden in a child class.

In [8]:
class MySequence(Sequence):
    def __init__(self, it: Iterable | None = None) -> None:
        self._seq = tuple(it or [])

    # sequence protocol
    def __getitem__(self, key: int | slice) -> Any:
        res = self._seq[key]
        if isinstance(key, int):
            return res
        return type(self)(res)

    # sequence protocol
    def __len__(self) -> int:
        return len(self.seq)

seq = MySequence()

# inheritance saves the situation
assert isinstance(seq, Sequence)

for item in MySequence([1, 2, 3]):
    print(f"Elem = {item}")

Elem = 1
Elem = 2
Elem = 3


Consider reading more about the [abc.Sequence](https://docs.python.org/3/library/collections.abc.html#collections.abc.Sequence) class.

## Iterator

Suppose we have some data structure that can be iterable not only in just one way. Imagine we have a 2D array (or list - whatever). A usual way is to walk through it row by row.

In [9]:
matrix = [[1], [2, 3], [4, 5, 6]]

# no need for an iterator here
for row in matrix:
    print(f"Row = {row}")

Row = [1]
Row = [2, 3]
Row = [4, 5, 6]


What if we need to iterate over its columns? Let's write a function for this.

In [10]:
T = TypeVar("T")

def iter_over_columns(mtx: list[list[T]]) -> Iterator[list[T]]:
    for ridx in range(len(mtx)):
        column: list[T] = []
        for row in mtx:
            if not row:
                continue

            item = None
            try:
                item = row[ridx]
            except IndexError:
                pass
            column.append(item)

        yield column
        

matrix = [[1], [2, 3], [4, 5, 6]]

# The `iter_over_columns` function is an iterator by nature,
# but it will fail the `isinstance` test on being Iterator.
for col in iter_over_columns(matrix):
    print(f"Column = {col}")

assert not isinstance(iter_over_columns, Iterator)

Column = [1, 2, 4]
Column = [None, 3, 5]
Column = [None, None, 6]


And what if we need other ways of traversion? It is doable with more iterators.

In [11]:
U = TypeVar("U")


class Matrix:
    def __init__(self, it: Iterable[Iterable[U]] | None = []) -> None:
        self._mtx: list[list[U]] = [list(row) for row in it] if it else []

    def __iter__(self) -> Iterator[U]:
        """The default strategy per rows."""

        for row in self._mtx:
            yield row

    def __reversed__(self) -> Iterator[U]:
        """The default strategy per rows."""

        for row in reversed(self._mtx):
            yield row


class MatrixColumnWalker:
    def __init__(self, matrix: Matrix) -> None:
        self._mtx = matrix

    def __iter__(self) -> Self:
        self.__it = self._traverse()
        return self

    def _traverse(self) -> Iterator[list[U]]:
        for ridx in range(len(matrix)):
            column: list[U] = []
            for row in self._mtx:
                if not row:
                    continue

                item = None
                try:
                    item = row[ridx]
                except IndexError:
                    pass
                column.append(item)

            yield column

    def __next__(self) -> Iterator[list[U]]:
        return next(self.__it)


class MatrixClockwiseWalker:
    """Up to you."""


class MatrixClounterClockwiseWalker:
    """Up to you."""


for matrix in (
    [],
    [[], []],
    [[1, 2], [3, 4]],
    [[1], [2, 3], [4, 5, 6]]  # not a good matrix
):
    print(f"Matrix: {matrix}")
    print(f"Per rows: {[row for row in matrix]}")
    print(f"Per rows reversed: {[row for row in reversed(matrix)]}")

    print(f"Per columns: {[row for row in MatrixColumnWalker(matrix)]}")

    print()

Matrix: []
Per rows: []
Per rows reversed: []
Per columns: []

Matrix: [[], []]
Per rows: [[], []]
Per rows reversed: [[], []]
Per columns: [[], []]

Matrix: [[1, 2], [3, 4]]
Per rows: [[1, 2], [3, 4]]
Per rows reversed: [[3, 4], [1, 2]]
Per columns: [[1, 3], [2, 4]]

Matrix: [[1], [2, 3], [4, 5, 6]]
Per rows: [[1], [2, 3], [4, 5, 6]]
Per rows reversed: [[4, 5, 6], [2, 3], [1]]
Per columns: [[1, 2, 4], [None, 3, 5], [None, None, 6]]



I was lazy enough to implement "Matrix[Counter]ClockWiseIterator"s to demonstrate other ways of traversing the matrix, but it is possible (I guess so). The same laziness did not encourage me to write "Tree" data structures from scratch to demonstrate that, e.g., "DepthIterator" and "BreadthIterator" could be useful when traversing a tree in depth or in breadth. Perhaps one day it will be done in my [ADS](https://github.com/stankudrow/ADS/tree/main/python) repository.

## Summary

Iterators are cool when:

- There are more than one way to traverse some collection. A default strategy may be implemented in the `__iter__()` method and, if needed, in the `__reversed__()` one.
- Iterators may be just functions, no need to make them classes.
- You don't need to overload a collection, you can just define an iterator that will make the collection to behave in the way you want to get the items. 